# **Purpose :**

Select face stimuli from the Chicago Face Database (CFD v3.0)
following Kardosh et al. (2022) criteria:
*   Rated as Black American by >= 90% of norming participants

*   Rated as White American by >= 90% of norming participants
*   330 images total, balanced by gender (50% male, 50% female)

* Copy selected images into minority_Faces/ and majority_Faces/

**Libraries imports**

### Python Equivalents for R Libraries

While there isn't a direct one-to-one mapping for every R package, here are the commonly used Python libraries that provide similar functionalities:

*   **`tidyverse`**: In Python, `pandas` is the go-to library for data manipulation and analysis, similar to `dplyr`. For plotting, `matplotlib` and `seaborn` are widely used, analogous to `ggplot2`.
*   **`here` / `fs`**: Python's `pathlib` module (part of the standard library) provides an object-oriented way to handle filesystem paths, making path construction and file operations clean and robust. The `os` module is also used for general operating system interactions, including file paths.
*   **`glue`**: Python has built-in f-strings (formatted string literals) for easy string interpolation, which serves a similar purpose to `glue`.
*   **`MatchIt` / `emmeans`**: These R packages are more specialized for causal inference matching and estimated marginal means. Python libraries like `statsmodels` can handle various statistical models and post-estimation, and `scikit-learn` provides general machine learning algorithms that can be adapted for some matching tasks. For specific causal inference methods, dedicated libraries like `DoWhy` or `CausalPy` might be relevant, though they don't directly mirror `MatchIt`'s API.

In [ ]:
# Install necessary libraries if you haven't already
# (Uncomment and run these lines if you encounter 'ModuleNotFoundError')
# !pip install pandas
# !pip install matplotlib
# !pip install seaborn
# !pip install statsmodels # For advanced statistical modeling and post-estimation

In [ ]:
from google.colab import drive
drive.flush_and_unmount()

Drive not mounted, so nothing to flush and unmount.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from pathlib import Path
import re
import numpy as np
from scipy.optimize import linear_sum_assignment
# For statistical modeling and related functionalities, if needed:
# import statsmodels.api as sm


**Mounting drive**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# **Data structure**

**Accessing the XLSX file**

In [ ]:
xlsx_path = Path("/content/drive/MyDrive/Colab_notebooks/Ulm/CFD_codebook.xlsx")

**Separating the sheets**


*   Codebook
*   Values



In [ ]:
codebook = pd.read_excel(xlsx_path, sheet_name="CFD 3.0 Codebook", header=11) # les titres des colonnes commencent en ligne 12
data = pd.read_excel(xlsx_path, sheet_name="CFD U.S. Norming Data")

/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


**Creating VarId --> Label mapping**

In [ ]:
codebook_clean = codebook[["VarID", "VarLabel", "Description"]].dropna(subset=["VarID"])
varid_to_label = dict(zip(codebook_clean["VarID"], codebook_clean["VarLabel"]))


**Charger les données + renommer les colonnes**
(se débarasser du multi-indexage sur les colonnes)

In [ ]:
data_raw = pd.read_excel(
    xlsx_path,
    sheet_name="CFD U.S. Norming Data",
    header=[6, 7, 8]   # lignes 7, 8, 9 dans Excel
)

data_raw.head()

Unnamed: 0_level_0               S001               S002               S003  \
               Model      EthnicitySelf         GenderSelf            AgeSelf   
  Unnamed: 0_level_2 Unnamed: 1_level_2 Unnamed: 2_level_2 Unnamed: 3_level_2   
0             AF-200                  A                  F                NaN   
1             AF-201                  A                  F                NaN   
2             AF-202                  A                  F                NaN   
3             AF-203                  A                  F                NaN   
4             AF-204                  A                  F                NaN   

        R002                       R003               R004               R005  \
    AgeRated                 FemaleProb           MaleProb          AsianProb   
   R002_mean R002_sd Unnamed: 6_level_2 Unnamed: 7_level_2 Unnamed: 8_level_2   
0  32.571429     NaN           1.000000           0.000000           1.000000   
1  23.666667     NaN           1.000000           0.000000           0.962963   
2  24.448276     NaN           0.827586           0.172414           0.310345   
3  22.758621     NaN           1.000000           0.000000           0.758621   
4  30.137931     NaN           1.000000           0.000000           0.827586   

               R005A  ...                 P056                 P057  \
    ChineseAsianProb  ...              EyeSize      UpperHeadLength   
  Unnamed: 9_level_2  ... Unnamed: 108_level_2 Unnamed: 109_level_2   
0                NaN  ...             0.060924             0.414099   
1                NaN  ...             0.041892             0.414414   
2                NaN  ...             0.051586             0.411080   
3                NaN  ...             0.063913             0.354407   
4                NaN  ...             0.053435             0.438931   

                  P058                 P059                 P060  \
         MidfaceLength           ChinLength       ForeheadHeight   
  Unnamed: 110_level_2 Unnamed: 111_level_2 Unnamed: 112_level_2   
0             0.326797             0.130719             0.264706   
1             0.329279             0.144595             0.300901   
2             0.310317             0.173424             0.298475   
3             0.343793             0.169820             0.272266   
4             0.293045             0.180237             0.293893   

                  P061                 P062                 P063  \
       CheekboneHeight  CheekboneProminence        FaceRoundness   
  Unnamed: 113_level_2 Unnamed: 114_level_2 Unnamed: 115_level_2   
0             0.388189                 91.5             0.545752   
1             0.383784                146.0             0.488288   
2             0.397029                 58.0             0.481333   
3             0.421089                 87.5             0.500231   
4             0.371925                 73.5             0.513571   

                  P065                 R001  
                 fWHR2               RaterN  
  Unnamed: 116_level_2 Unnamed: 117_level_2  
0             1.973890                   28  
1             2.068681                   27  
2             1.987261                   29  
3             1.902284                   29  
4             2.081081                   29  

[5 rows x 118 columns]

In [ ]:
def clean_column(col):
    """
    col est un tuple du type :
    ('S001', 'EthnicitySelf', 'Unnamed: ...')
    ou
    ('R002', 'AgeRated', 'R002_mean')
    """

    varid, varlabel, sublabel = col

    varid = str(varid)
    varlabel = str(varlabel)
    sublabel = str(sublabel)

    #colonne Model
    if varlabel == "Model":
        return "Model"

    #si sous-variable explicite : R002_mean, R002_sd, etc.
    if not sublabel.startswith("Unnamed"):
        if "_" in sublabel:
            suffix = sublabel.split("_", 1)[1]
            return f"{varlabel}_{suffix}"
        else:
            return sublabel

    # Cas normal : S001 -> EthnicitySelf, S002 -> GenderSelf, etc.
    return varlabel


data = data_raw.copy()
data.columns = [clean_column(col) for col in data_raw.columns]

data.head()
'''
NB : c'est normal qu'il n'y ait que 5 lignes affichées, si on éxécute data.shape on a bien 597 lignes/profils
'''


"\nNB : c'est normal qu'il n'y ait que 5 lignes affichées, si on éxécute data.shape on a bien 597 lignes/profils\n"

**Liste des infos extractibles par image :**

In [ ]:
data.columns.tolist()

['Model',
 'EthnicitySelf',
 'GenderSelf',
 'AgeSelf',
 'AgeRated_mean',
 'AgeRated_sd',
 'FemaleProb',
 'MaleProb',
 'AsianProb',
 'ChineseAsianProb',
 'JapaneseAsianProb',
 'IndianAsianProb',
 'OtherAsianProb',
 'MiddleEasternProb',
 'BlackProb',
 'LatinoProb',
 'MultiProb',
 'OtherProb',
 'WhiteProb',
 'Afraid_mean',
 'Afraid_sd',
 'Angry_mean',
 'Angry_sd',
 'Attractive_mean',
 'Attractive_sd',
 'Babyfaced_mean',
 'Babyfaced_sd',
 'Disgusted_mean',
 'Disgusted_sd',
 'Dominant_mean',
 'Dominant_sd',
 'Feminine_mean',
 'Feminine_sd',
 'Happy_mean',
 'Happy_sd',
 'Masculine_mean',
 'Masculine_sd',
 'Prototypic_mean',
 'Prototypic_sd',
 'Sad_mean',
 'Sad_sd',
 'Suitability_mean',
 'Suitability_sd',
 'Surprised_mean',
 'Surprised_sd',
 'Threatening_mean',
 'Threatening_sd',
 'Trustworthy_mean',
 'Trustworthy_sd',
 'Unusual_mean',
 'Unusual_sd',
 'Warm_mean',
 'Warm_sd',
 'Competent_mean',
 'Competent_sd',
 'SocialStatus_mean',
 'SocialStatus_sd',
 'LuminanceMedian',
 'NoseWidth',
 'Nose

# **Link to images**

Les images sont une-à-une dans le dossier nommé par l'id du modèle (très très chiant comme format).

ATTENTION À LA NEUTRALITÉ DU VISAGE (À RETRAVAILLER)

In [ ]:
images_dir= Path("/content/drive/MyDrive/Colab_notebooks/Ulm/Images/CFD")

IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png",
    ".tif", ".tiff",
    ".bmp", ".webp"
}


def clean_drive_suffix(name):
    # enlève les suffixes type " (1)", " (2)", etc.
    return re.sub(r"\s\(\d+\)$", "", str(name).strip())

# Dictionnaire : nom nettoyé -> vrai dossier
folder_map = {}

for folder in images_dir.iterdir():
    if folder.is_dir():
        clean_name = clean_drive_suffix(folder.name)
        folder_map[clean_name] = folder

def find_model_image(model_id):
    model_id = str(model_id).strip()

    model_dir = folder_map.get(model_id)

    if model_dir is None:
        return None

    image_files = [
        f for f in model_dir.rglob("*")
        if f.is_file()
        and not f.name.startswith(".")
        and f.suffix.lower() in IMAGE_EXTENSIONS
    ]

    candidates = []
    for ext in ["*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"]:
        candidates.extend(model_dir.glob(ext))


    # Priorité aux images neutres si le nom contient N ou Neutral
    neutral = [
        f for f in candidates
        if "N" in f.name.lower() or "_N" in f.name.lower()
    ]

    if neutral:
        return neutral[0]

    if candidates:
        return candidates[0]

    if len(image_files) == 0:
        return None

    return str(image_files[0])

data["image_path"] = data["Model"].astype(str).str.strip().apply(find_model_image)
data["image_exists"] = data["image_path"].notna()

data["image_exists"].value_counts()

,count
image_exists,
True,597


In [ ]:
data["image_path"] = data["Model"].apply(find_model_image)
data["image_exists"] = data["image_path"].notna()

In [ ]:
data[["Model", "image_path", "image_exists"]].head(20)

,Model,image_path,image_exists
0,AF-200,/content/drive/MyDrive/Colab_notebooks/Ulm/Ima...,True
1,AF-201,/content/drive/MyDrive/Colab_notebooks/Ulm/Ima...,True
2,AF-202,/content/drive/MyDrive/Colab_notebooks/Ulm/Ima...,True
3,AF-203,/content/drive/MyDrive/Colab_notebooks/Ulm/Ima...,True
4,AF-204,/content/drive/MyDrive/Colab_notebooks/Ulm/Ima...,True
5,AF-205,/content/drive/MyDrive/Colab_notebooks/Ulm/Ima...,True
6,AF-206,/content/drive/MyDrive/Colab_notebooks/Ulm/Ima...,True
7,AF-207,/content/drive/MyDrive/Colab_notebooks/Ulm/Ima...,True
8,AF-208,/content/drive/MyDrive/Colab_notebooks/Ulm/Ima...,True
9,AF-209,/content/drive/MyDrive/Colab_notebooks/Ulm/Ima...,True


**Confirmation qu'on a bien tous les modèles**

In [ ]:
data["image_exists"].value_counts()

,count
image_exists,
True,597


# **APPLY SELECTION CRITERIA**

## Selection criteria: Black/White × Gender candidates

Equivalent Python/pandas version of Lucile's R filtering code.
This starts from `data`, keeps only rows with an available image, then creates `PRace`, `PGender`, and `cond`.


 **Parameters** :

In [ ]:
PROB_THRESHOLD= 0.90   # >= 90% rated as target ethnicity (paper criterion)
PROP_GENDER= 0.50   # proportion female within each ethnic group
SEED= 2022   # for reproducibility


In [ ]:
# Start from the cleaned dataframe created above
cfd = data.copy()

# Keep only rows with an image that was actually found
cfd = cfd[cfd["image_exists"]].copy()

# Make sure probability columns are numeric
prob_cols = ["BlackProb", "WhiteProb", "FemaleProb", "MaleProb"]
for col in prob_cols:
    cfd[col] = pd.to_numeric(cfd[col], errors="coerce")

print("Rows with available images:", len(cfd))
print(cfd[prob_cols].describe())


Rows with available images: 597
        BlackProb   WhiteProb  FemaleProb    MaleProb
count  597.000000  597.000000  597.000000  597.000000
mean     0.289099    0.315151    0.509534    0.490466
std      0.427225    0.419765    0.491784    0.491784
min      0.000000    0.000000    0.000000    0.000000
25%      0.000000    0.000000    0.000000    0.000000
50%      0.000000    0.011236    0.913043    0.086957
75%      0.884615    0.840000    1.000000    1.000000
max      1.000000    1.000000    1.000000    1.000000


### Count candidate images per condition


In [ ]:
# Equivalent of:
# filter((BlackProb >= PROB_THRESHOLD & FemaleProb >= 0.9) | ...)
mask_candidates = (
    ((cfd["BlackProb"] >= PROB_THRESHOLD) & (cfd["FemaleProb"] >= PROB_THRESHOLD)) |
    ((cfd["BlackProb"] >= PROB_THRESHOLD) & (cfd["MaleProb"] >= PROB_THRESHOLD)) |
    ((cfd["WhiteProb"] >= PROB_THRESHOLD) & (cfd["FemaleProb"] >= PROB_THRESHOLD)) |
    ((cfd["WhiteProb"] >= PROB_THRESHOLD) & (cfd["MaleProb"] >= PROB_THRESHOLD))
)

n_candidates = cfd.loc[mask_candidates].copy()

# PRace: B = perceived Black, W = perceived White
n_candidates["PRace"] = np.where(
    n_candidates["BlackProb"] >= PROB_THRESHOLD,
    "B",
    "W"
)

# PGender: W = woman, M = man
n_candidates["PGender"] = np.where(
    n_candidates["FemaleProb"] >= PROB_THRESHOLD,
    "W",
    "M"
)

# Keep all combinations, even if one condition has zero candidates
all_conditions = pd.MultiIndex.from_product(
    [["W", "B"], ["M", "W"]],
    names=["PRace", "PGender"]
)

n_candidates_count = (
    n_candidates
    .groupby(["PRace", "PGender"])
    .size()
    .reindex(all_conditions, fill_value=0)
    .reset_index(name="n")
)

n_candidates_count["condition"] = (
    n_candidates_count["PRace"] + n_candidates_count["PGender"]
)

# Same condition order as your R code
condition_order = ["WM", "BM", "WW", "BW"]
n_candidates_count["condition"] = pd.Categorical(
    n_candidates_count["condition"],
    categories=condition_order,
    ordered=True
)

n_candidates_count = n_candidates_count.sort_values("condition").reset_index(drop=True)

display(n_candidates_count)

n = int(n_candidates_count["n"].min())
print(f"Maximum number of images per condition: {n}")


,PRace,PGender,n,condition
0,W,M,74,WM
1,B,M,61,BM
2,W,W,61,WW
3,B,W,79,BW


Maximum number of images per condition: 61


### Create the filtered CFD dataframe with condition labels
Garder les lignes où :
la race perçue est clairement Black ou White
ET
le genre perçu est clairement Female ou Male


In [ ]:
# Equivalent of:
# filter((BlackProb >= PROB_THRESHOLD | WhiteProb >= PROB_THRESHOLD) &
#        (FemaleProb >= PROB_THRESHOLD | MaleProb >= PROB_THRESHOLD))
mask = (
    ((cfd["BlackProb"] >= PROB_THRESHOLD) | (cfd["WhiteProb"] >= PROB_THRESHOLD)) &
    ((cfd["FemaleProb"] >= PROB_THRESHOLD) | (cfd["MaleProb"] >= PROB_THRESHOLD))
)

cfd_selected = cfd.loc[mask].copy()

cfd_selected["PRace"] = np.where(
    cfd_selected["BlackProb"] >= PROB_THRESHOLD,
    "B",
    "W"
)

cfd_selected["PGender"] = np.where(
    cfd_selected["FemaleProb"] >= PROB_THRESHOLD,
    "W",
    "M"
)

cfd_selected["cond"] = cfd_selected["PRace"] + cfd_selected["PGender"]

cfd_selected["cond"] = pd.Categorical(
    cfd_selected["cond"],
    categories=condition_order,
    ordered=True
)

print(cfd_selected.shape)
display(cfd_selected[[
    "Model", "PRace", "PGender", "cond",
    "BlackProb", "WhiteProb", "FemaleProb", "MaleProb",
    "image_path"
]].head())


(275, 123)


,Model,PRace,PGender,cond,BlackProb,WhiteProb,FemaleProb,MaleProb,image_path
109,BF-001,B,W,BW,1.000000,0.0,1.000000,0.000000,/content/drive/MyDrive/Colab_notebooks/Ulm/Ima...
110,BF-002,B,W,BW,0.978495,0.0,1.000000,0.000000,/content/drive/MyDrive/Colab_notebooks/Ulm/Ima...
112,BF-004,B,W,BW,1.000000,0.0,1.000000,0.000000,/content/drive/MyDrive/Colab_notebooks/Ulm/Ima...
114,BF-006,B,W,BW,0.957895,0.0,0.989362,0.010638,/content/drive/MyDrive/Colab_notebooks/Ulm/Ima...
115,BF-007,B,W,BW,0.978261,0.0,0.978261,0.021739,/content/drive/MyDrive/Colab_notebooks/Ulm/Ima...


### Save the whole table that respects the conditions


In [ ]:
output_dir = Path("/content/drive/MyDrive/Colab_notebooks/Ulm/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

selected_path = output_dir / "cfd_selected_with_conditions.csv"
balanced_path = output_dir / "cfd_balanced_with_conditions.csv"

cfd_selected.to_csv(selected_path, index=False)
'''
if len(cfd_balanced) > 0:
    cfd_balanced.to_csv(balanced_path, index=False)
'''
print("Saved:", selected_path)
'''
if len(cfd_balanced) > 0:
    print("Saved:", balanced_path)
'''

NameError: name 'cfd_balanced' is not defined

Faire en sorte que les groupes aient des distributions d’âge similaires.

In [ ]:
# On repart du dataframe déjà filtré avec les colonnes PRace, PGender, cond
cfd = cfd_selected.copy()

# Si l'âge est dans AgeRated_mean, on l'utilise.
# Si on a une colonne AgeRated, le code l'utilise directement.
AGE_COL = "AgeRated" if "AgeRated" in cfd.columns else "AgeRated_mean"

print("Colonne d'âge utilisée :", AGE_COL)

# On enlève les lignes sans âge
cfd = cfd.dropna(subset=[AGE_COL, "cond"]).copy()

# Ordre des conditions
cond_order = ["WM", "BM", "WW", "BW"]

cfd["cond"] = pd.Categorical(
    cfd["cond"],
    categories=cond_order,
    ordered=True
)

cfd["cond"].value_counts().sort_index()

Colonne d'âge utilisée : AgeRated_mean


,count
cond,
WM,74
BM,61
WW,61
BW,79


Le matching sert  à dire :

mes groupes diffèrent sur la condition expérimentale, mais pas trop sur l’âge perçu.

In [ ]:
def match_pair(data, cond_a, cond_b, age_col=AGE_COL):
    """
    Équivalent Python de ta fonction R match_pair.

    On prend deux conditions, par exemple BM et WW.
    On fait un matching optimal sur l'âge.
    On retourne seulement les modèles de cond_a sélectionnés après matching.
    """

    group_a = data[data["cond"] == cond_a].copy().reset_index(drop=False)
    group_b = data[data["cond"] == cond_b].copy().reset_index(drop=False)

    if len(group_a) == 0 or len(group_b) == 0:
        raise ValueError(f"Un des groupes est vide : {cond_a}={len(group_a)}, {cond_b}={len(group_b)}")

    # Matrice de coût : différence absolue d'âge entre chaque paire A-B
    cost_matrix = np.abs(
        group_a[age_col].to_numpy()[:, None] -
        group_b[age_col].to_numpy()[None, :]
    )

    # Matching optimal
    row_ind, col_ind = linear_sum_assignment(cost_matrix)

    # On garde les lignes de group_a qui ont été matchées
    matched_a = group_a.iloc[row_ind].copy()

    # Optionnel : infos de matching
    matched_a["matched_with"] = group_b.iloc[col_ind]["Model"].to_numpy()
    matched_a["age_distance"] = cost_matrix[row_ind, col_ind]

    # On remet l'ancien index comme index pandas
    matched_a = matched_a.set_index("index")

    return matched_a


#SAMPLE: EQUAL N PER GROUP, BALANCED GENDER


In [ ]:
# step 1 : match BM to WW (most constrained pair, n = 61 each)

matched_BM = match_pair(cfd, "BM", "WW")
matched_WW = match_pair(cfd, "WW", "BM")

print("matched_BM:", matched_BM.shape)
print("matched_WW:", matched_WW.shape)

# step 2 : match WM to the already-selected BM models

tmp_WM_BM = pd.concat(
    [
        cfd[cfd["cond"] == "WM"],
        matched_BM[cfd.columns]
    ],
    axis=0
).copy()

matched_WM = match_pair(tmp_WM_BM, "WM", "BM")

print("matched_WM:", matched_WM.shape)

# Step 3: match BW to the already-selected WW models


tmp_BW_WW = pd.concat(
    [
        cfd[cfd["cond"] == "BW"],
        matched_WW[cfd.columns]
    ],
    axis=0
).copy()

matched_BW = match_pair(tmp_BW_WW, "BW", "WW")

print("matched_BW:", matched_BW.shape)



#######################
final_stimuli = pd.concat(
    [
        matched_BM[cfd.columns],
        matched_WM[cfd.columns],
        matched_WW[cfd.columns],
        matched_BW[cfd.columns]
    ],
    axis=0
).copy()

final_stimuli["cond"] = pd.Categorical(
    final_stimuli["cond"],
    categories=["WM", "BM", "WW", "BW"],
    ordered=True
)

final_stimuli["cond"].value_counts().sort_index()

matched_BM: (61, 125)
matched_WW: (61, 125)
matched_WM: (61, 125)
matched_BW: (61, 125)


,count
cond,
WM,61
BM,61
WW,61
BW,61


THIS PART OF R CODE TO CHECK :  ggplot(data = final_stimuli, aes(x = cond, y = AgeRated)) +
    geom_boxplot()
  
  
  model <- lm(data = final_stimuli,
              AgeRated ~ cond)
  
  summary(model)
  
  emm <- emmeans(model,
                 specs = pairwise ~ cond,
                 adjust="holm") # to compare within categories of emotion
  emm

# **COPY FINAL SELECTED IMAGES**

In [ ]:
CFD_IMG_ROOT = Path("/content/drive/MyDrive/Colab_notebooks/Ulm/Images/CFD")

OUT_MIN = Path("/content/drive/MyDrive/Colab_notebooks/Ulm/minority_Faces")
OUT_MAJ = Path("/content/drive/MyDrive/Colab_notebooks/Ulm/majority_Faces")

# Create output folders
OUT_MIN.mkdir(parents=True, exist_ok=True)
OUT_MAJ.mkdir(parents=True, exist_ok=True)


# Locate, assign destination, and copy
copied_black = 0
copied_white = 0

for _, row in final_stimuli.iterrows():
    model = str(row["Model"]).strip()
    prace = row["PRace"]

    model_dir = CFD_IMG_ROOT / model

    # équivalent de dir_ls(path(CFD_IMG_ROOT, .x), regexp = "-N\\.jpg$")
    hits = list(model_dir.rglob("*-N.jpg"))

    if len(hits) == 0:
        warnings.warn(f"No image found for: {model}")
        continue

    image_path = hits[0]

    # équivalent de if_else(PRace == "B", OUT_MIN, OUT_MAJ)
    if prace == "B":
        dest_dir = OUT_MIN
        copied_black += 1
    else:
        dest_dir = OUT_MAJ
        copied_white += 1

    dest_path = dest_dir / f"{model}_neutral.jpg"

    # équivalent de file_copy(..., overwrite = TRUE)
    shutil.copy2(image_path, dest_path)


print(
    f"Copied {copied_black} Black → minority_Faces/\n"
    f"Copied {copied_white} White → majority_Faces/"
)

Copied 122 Black → minority_Faces/
Copied 122 White → majority_Faces/
